## **IMPORTS**

In [1]:
# Apache Spark API
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import lit
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import NaiveBayes, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Data Visualization
import matplotlib.pyplot as plt

In [2]:
# Iniciando Spark Session
spark = SparkSession.builder \
    .appName("BigData_AC2") \
    .master("local[*]") \
    .config("spark.executor.memory", "12g") \
    .config("spark.driver.memory",   "12g") \
    .config("spark.cleaner.referenceTracking.blocking", "true") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.storage.cleanupFilesAfterExecutorExit", "true") \
    .getOrCreate()

In [3]:
# Definição de Schema dos dados
train = spark.read.parquet("../../../data/train.parquet", header=True)
print(f"train N° samples: {train.count()}")

train N° samples: 46029416


In [4]:
spark.catalog.clearCache()
spark.sql("CLEAR CACHE")

DataFrame[]

## **DATA TREATMENT**

In [5]:
assembler = VectorAssembler(
    inputCols=[
         'DepTime',
         'CRSDepTime',
         'ArrTime',
         'CRSArrTime',
         'ActualElapsedTime',
         'CRSElapsedTime',
         'AirTime',
         'ArrDelay',
         'DepDelay',
         'Distance',
         'TaxiIn',
         'TaxiOut',
         'Month_1',
         'Month_10',
         'Month_11',
         'Month_12',
         'Month_2',
         'Month_3',
         'Month_4',
         'Month_5',
         'Month_6',
         'Month_7',
         'Month_8',
         'Month_9'
    ], outputCol='features',
)
input_train = assembler.transform(train)

## **MULTILAYER PERCEPTRON CLASSIFIER**

### TRAINING

In [6]:
ml = MultilayerPerceptronClassifier(
    labelCol='isDelay',
    featuresCol='features',
    layers=[24, 3, 2],
    blockSize=128,
    stepSize=0.03,
    maxIter=13,
    seed=2025
)
model = ml.fit(input_train)

In [7]:
test = spark.read.parquet("../../../data/test.parquet", header=True)
print(f"test N° samples: {test.count()}")
input_test = assembler.transform(test)

test N° samples: 14653974


### PREDICTING AND VALIDATION

In [8]:
predictions = model.transform(input_test)

In [9]:
evaluator = MulticlassClassificationEvaluator(labelCol="isDelay", metricName="accuracy")
print("Accuracy:", evaluator.evaluate(predictions))

Accuracy: 0.9820983031633603


In [10]:
evaluator = MulticlassClassificationEvaluator(labelCol="isDelay", metricName="precisionByLabel")
print("Precision by Label:", evaluator.evaluate(predictions))

Precision by Label: 0.9917572010535036


In [11]:
evaluator = MulticlassClassificationEvaluator(labelCol="isDelay", metricName="recallByLabel")
print("Recall by Label:", evaluator.evaluate(predictions))

Recall by Label: 0.9783436326957653


## **GAUSSIAN NAIVE BAYS**

### TRAINING

In [ ]:
nb = NaiveBayes(
    labelCol='isDelay',
    featuresCol='features',
    smoothing = 1.0,
    modelType = 'gaussian'
)
model_nb = nb.fit(input_train)

### PREDICTING AND VALIDATION

In [15]:
predictions_nb = model_nb.transform(input_test)

In [16]:
evaluator = MulticlassClassificationEvaluator(labelCol="isDelay", metricName="accuracy")
print("Accuracy:", evaluator.evaluate(predictions_nb))

Accuracy: 0.9198433817338559


In [17]:
evaluator = MulticlassClassificationEvaluator(labelCol="isDelay", metricName="precisionByLabel")
print("Precision by Label:", evaluator.evaluate(predictions_nb))

Precision by Label: 0.9104424331018739


In [18]:
evaluator = MulticlassClassificationEvaluator(labelCol="isDelay", metricName="recallByLabel")
print("Recall by Label:", evaluator.evaluate(predictions_nb))

Recall by Label: 0.9611702189266483
